In [ ]:
from pyqueueing import MM1, MMC, MM1K, MMcK, MG1, ErlangB

## 1. Impact of Service Time Variability (M/G/1)

The Pollaczek-Khinchine formula shows that Lq is proportional to (1 + Cs²).
Higher variability → longer queues.

In [ ]:
lam, mu = 0.8, 1.0  # ρ = 0.8

print(f"{'Service Type':<20} {'CV':>5} {'Lq':>8} {'Wq':>8} {'W':>8}")
print("-" * 52)

cases = [
    ("Deterministic", 0.0),
    ("Low variance", 0.5),
    ("Exponential", 1.0),
    ("High variance", 1.5),
    ("Very high var.", 2.0),
]
for name, cv in cases:
    q = MG1(arrival_rate=lam, service_rate=mu, service_cv=cv)
    print(f"{name:<20} {cv:>5.1f} {q.mean_queue_length():>8.4f} {q.mean_wait():>8.4f} {q.mean_system_time():>8.4f}")

## 2. Finite vs Infinite Capacity

How does buffer size affect packet loss and delay?

In [ ]:
lam, mu = 0.9, 1.0  # ρ = 0.9

mm1 = MM1(arrival_rate=lam, service_rate=mu)
print(f"M/M/1 (infinite):  L={mm1.mean_system_size():.4f}, W={mm1.mean_system_time():.4f}")
print()
print(f"{'K':>4} {'P(block)':>10} {'L':>8} {'W':>8} {'λ_eff':>8}")
print("-" * 42)
for K in [5, 10, 20, 50, 100]:
    q = MM1K(arrival_rate=lam, service_rate=mu, capacity=K)
    print(f"{K:>4d} {q.prob_block():>10.6f} {q.mean_system_size():>8.4f} {q.mean_system_time():>8.4f} {q.effective_arrival_rate():>8.4f}")

## 3. Adding Servers vs Adding Buffer

Which is more cost-effective: more servers or more buffer?

In [ ]:
lam, mu = 10.0, 3.0

print("--- More servers (M/M/c, K=∞) ---")
print(f"{'c':>3} {'ρ':>6} {'P(wait)':>8} {'Wq':>8} {'Lq':>8}")
for c in range(4, 8):
    q = MMC(arrival_rate=lam, service_rate=mu, servers=c)
    print(f"{c:>3d} {q.utilization():>6.2%} {q.prob_wait():>8.4f} {q.mean_wait():>8.4f} {q.mean_queue_length():>8.4f}")

print()
print("--- More buffer (M/M/4/K) ---")
print(f"{'K':>3} {'P(block)':>9} {'Wq':>8} {'Lq':>8}")
for K in [4, 8, 12, 20]:
    q = MMcK(arrival_rate=lam, service_rate=mu, servers=4, capacity=K)
    print(f"{K:>3d} {q.prob_block():>9.4%} {q.mean_wait():>8.4f} {q.mean_queue_length():>8.4f}")

## 4. Model Equivalence Checks

Sanity checks to verify the models are consistent.

In [ ]:
import math

lam, mu = 2.0, 3.0

# MMC(c=1) should equal MM1
mm1 = MM1(arrival_rate=lam, service_rate=mu)
mmc1 = MMC(arrival_rate=lam, service_rate=mu, servers=1)
print("MMC(c=1) == MM1?")
print(f"  Lq: {mm1.mean_queue_length():.6f} vs {mmc1.mean_queue_length():.6f}  ✓" 
      if math.isclose(mm1.mean_queue_length(), mmc1.mean_queue_length()) 
      else f"  Lq: MISMATCH!")

# MG1(cv=1) should equal MM1
mg1 = MG1(arrival_rate=lam, service_rate=mu, service_cv=1.0)
print(f"\nMG1(cv=1) == MM1?")
print(f"  Lq: {mm1.mean_queue_length():.6f} vs {mg1.mean_queue_length():.6f}  ✓")

# MM1K(K=large) should approximate MM1
mm1k = MM1K(arrival_rate=lam, service_rate=mu, capacity=100)
print(f"\nMM1K(K=100) ≈ MM1?")
print(f"  L: {mm1.mean_system_size():.6f} vs {mm1k.mean_system_size():.6f}  ✓")

# MMcK(K=c) should equal ErlangB
c = 10
mmck = MMcK(arrival_rate=lam, service_rate=mu, servers=c, capacity=c)
eb = ErlangB(arrival_rate=lam, service_rate=mu, servers=c)
print(f"\nMMcK(K=c) == ErlangB?")
print(f"  P(block): {mmck.prob_block():.8f} vs {eb.prob_block():.8f}  ✓")